# Strategy Comparison — Manual vs ML Equity Curves
Load backtest results from the QuantEdge API or local JSON files, then compare
manual and ML-enhanced strategy performance with benchmark overlays.

In [ ]:
!pip install -q pandas numpy matplotlib requests

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────────────────────
API_BASE     = 'http://localhost:8000/api/v1'   # Update if remote
STRATEGY     = 'momentum'
SYMBOL       = 'SPY'
USE_API      = False  # Set True if QuantEdge backend is running
DATA_DIR     = Path('../pipeline_runs.json')    # Fallback local data

plt.rcParams.update({
    'figure.facecolor': '#131722',
    'axes.facecolor':   '#1e2433',
    'axes.edgecolor':   '#ffffff22',
    'text.color':       '#cccccc',
    'axes.labelcolor':  '#888888',
    'xtick.color':      '#888888',
    'ytick.color':      '#888888',
    'grid.color':       '#ffffff0d',
    'grid.linestyle':   '--',
})
print('Config loaded')

In [ ]:
import requests

def load_comparison_data():
    if USE_API:
        try:
            resp = requests.get(
                f'{API_BASE}/comparison/results',
                params={'strategy': STRATEGY, 'symbol': SYMBOL},
                timeout=10
            )
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            print(f'API call failed: {e} — using synthetic demo data')

    # Generate synthetic equity curves for demo
    np.random.seed(42)
    dates = pd.date_range('2023-01-01', periods=252, freq='B')

    manual_rets = np.random.normal(0.0003, 0.008, 252)
    ml_rets     = np.random.normal(0.0007, 0.007, 252)  # slightly better

    manual_equity = 100 * np.cumprod(1 + manual_rets)
    ml_equity     = 100 * np.cumprod(1 + ml_rets)

    # SPY benchmark
    spy_rets   = np.random.normal(0.0004, 0.009, 252)
    spy_equity = 100 * np.cumprod(1 + spy_rets)

    def sharpe(rets, ann=252):
        return (rets.mean() / (rets.std() + 1e-9)) * np.sqrt(ann)

    def max_dd(equity):
        roll_max = np.maximum.accumulate(equity)
        return ((equity - roll_max) / roll_max).min() * 100

    return {
        'dates':         [str(d.date()) for d in dates],
        'manual_equity': manual_equity.tolist(),
        'ml_equity':     ml_equity.tolist(),
        'spy_equity':    spy_equity.tolist(),
        'manual_metrics': {
            'sharpe':           round(float(sharpe(manual_rets)), 3),
            'return_pct':       round(float((manual_equity[-1]/100 - 1)*100), 2),
            'max_drawdown_pct': round(float(max_dd(manual_equity)), 2),
            'win_rate':         round(float((manual_rets > 0).mean() * 100), 1),
            'num_trades':       189,
        },
        'ml_metrics': {
            'sharpe':           round(float(sharpe(ml_rets)), 3),
            'return_pct':       round(float((ml_equity[-1]/100 - 1)*100), 2),
            'max_drawdown_pct': round(float(max_dd(ml_equity)), 2),
            'win_rate':         round(float((ml_rets > 0).mean() * 100), 1),
            'num_trades':       231,
        },
        'spy_metrics': {
            'sharpe':           round(float(sharpe(spy_rets)), 3),
            'return_pct':       round(float((spy_equity[-1]/100 - 1)*100), 2),
        },
    }

data = load_comparison_data()
print('Metrics loaded:')
print(f"  Manual   — Sharpe: {data['manual_metrics']['sharpe']:.3f} | Return: {data['manual_metrics']['return_pct']:.1f}%")
print(f"  ML Enh   — Sharpe: {data['ml_metrics']['sharpe']:.3f} | Return: {data['ml_metrics']['return_pct']:.1f}%")
print(f"  SPY      — Sharpe: {data['spy_metrics']['sharpe']:.3f} | Return: {data['spy_metrics']['return_pct']:.1f}%")

In [ ]:
dates          = pd.to_datetime(data['dates'])
manual_equity  = np.array(data['manual_equity'])
ml_equity      = np.array(data['ml_equity'])
spy_equity     = np.array(data['spy_equity'])

# Equity curve overlay
fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [3, 1]})

ax = axes[0]
ax.plot(dates, manual_equity, color='#f5a623', linewidth=1.5, label=f"Manual (Sharpe {data['manual_metrics']['sharpe']:.2f})")
ax.plot(dates, ml_equity,     color='#00c853', linewidth=1.5, label=f"ML Enhanced (Sharpe {data['ml_metrics']['sharpe']:.2f})")
ax.plot(dates, spy_equity,    color='#2979ff', linewidth=1.0, linestyle='--', alpha=0.6, label='SPY Benchmark')
ax.fill_between(dates, manual_equity, spy_equity, where=manual_equity >= spy_equity, alpha=0.07, color='#f5a623')
ax.fill_between(dates, ml_equity,     spy_equity, where=ml_equity >= spy_equity,     alpha=0.07, color='#00c853')
ax.set_title(f'Equity Curves — {STRATEGY.upper()} / {SYMBOL}', fontsize=13, color='white', pad=10)
ax.set_ylabel('Portfolio Value ($)', fontsize=10)
ax.legend(loc='upper left', framealpha=0.1, fontsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Drawdown panel
ax2 = axes[1]
def drawdown(equity):
    roll_max = np.maximum.accumulate(equity)
    return (equity - roll_max) / roll_max * 100

ax2.fill_between(dates, drawdown(manual_equity), alpha=0.3, color='#f5a623', label='Manual DD')
ax2.fill_between(dates, drawdown(ml_equity),     alpha=0.3, color='#00c853', label='ML DD')
ax2.set_ylabel('Drawdown %', fontsize=9)
ax2.set_ylim(top=2)
ax2.legend(loc='lower left', framealpha=0.1, fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.tight_layout()
plt.savefig('strategy_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved to strategy_comparison.png')

In [ ]:
# Metrics comparison table
metrics = ['sharpe', 'return_pct', 'max_drawdown_pct', 'win_rate', 'num_trades']
labels  = ['Sharpe Ratio', 'Return %', 'Max Drawdown %', 'Win Rate %', 'Num Trades']

print('\n' + '='*55)
print(f'{"Metric":<22} {"Manual":>12} {"ML Enhanced":>14}')
print('='*55)
for metric, label in zip(metrics, labels):
    mv = data['manual_metrics'].get(metric)
    xv = data['ml_metrics'].get(metric)
    mstr = f'{mv:.2f}' if mv is not None else '—'
    xstr = f'{xv:.2f}' if xv is not None else '—'
    winner = '★ ML' if (xv or 0) > (mv or 0) else '★ MN' if (mv or 0) > (xv or 0) else '  TIE'
    print(f'{label:<22} {mstr:>12} {xstr:>14}  {winner}')
print('='*55)